# Assignment 2- Methods and Plan & Computational Code and Output

Submitted by- Antarip Kashyap (20118782) in Group 33 "301 Byte"

Our group is assigned the dataset- Wildfire from Diverse Data in R packages.

In this notebook, I will be attemping to plan out the methods and plan computational Code and output following on from the EDA that I did on this dataset in assignment 1 to create a comprehensive plan to undertake our upcoming group project.

We will begin by downloading the dataset and having a look at what it has.

This assignment starts from the EDA and we then lead on to the methods section. Sections that have a clear change from assignment 1 are highlighted with a preceeding markdown block.

# Assignment 1 materials with corrections: Planning Stage: Data Description & Exploratory Data Analysis

In [ ]:
#Please uncomment the following line before re-running the notebook
devtools::install_github("diverse-data-hub/diversedata")

In [ ]:
library(diversedata)
library(dplyr)
library(ggplot2)
library(dplyr)
library(broom)
library(car)
library(lmtest)

?wildfire

In [ ]:
data("wildfire")

In [ ]:
head(wildfire)

In [ ]:
dim(wildfire)

## (1) Data Description

- The Data set has 35 variables with 26,551 observations.
- The following code cell is a table with all the names, and types of variables

- The dataset was provided under the Open Government Licence - Alberta (according to the dataset help text)  by the Government of Alberta, Canada.
- The dataset has an Open Government Liscense from Alberta. According to the liscensing page associated with this type of license,  the following terms must specifically be adhered to-

*"The Information Provider grants you a worldwide, royalty-free, perpetual, non-exclusive licence to use the Information, including for commercial purposes, subject to the terms below."*

*"You are free to:
Copy, modify, publish, translate, adapt, distribute or otherwise use the Information in any medium, mode or format for any lawful purpose.
You must, where you do any of the above:*

**Acknowledge the source of the Information by including any attribution statement specified by the Information Provider(s) and, where possible, provide a link to this licence.
If the Information Provider does not provide a specific attribution statement, or if you are using Information from several Information Providers and multiple attributions are not practical for your product or application, you must use the following attribution statement:**

                 Contains information licensed under the Open Government Licence – Alberta.

*The terms of this licence are important, and if you fail to comply with any of them, the rights granted to you under this licence, or any similar licence granted by the Information Provider, will end automatically."*


> More important the following exemptions must also be ahdered to-

*"This licence does not grant you any right to use:
Personal Information;
Information or Records that are not accessible under applicable laws;
third party rights the Information Provider is not authorized to license;
the names, crests, logos, or other official symbols of the Information Provider; and
Information subject to other intellectual property rights, including patents, trade-marks and official marks."*

In [ ]:
variable_table <- data.frame(
  variable_type = sapply(wildfire, function(x) paste(class(x), collapse = ", "))
)

variable_table

## (2) Questions

In this section, I will be using it to generate a question to propse to my group that we could use for our group project.

- I want to study how environmental, seasonal, and geographic conditions jointly explain variation in wildfire size using a multiple linear regression approach. The aim here is to ask which combination of wildfire related factors are associated with larger fires.

- This question focuses on both inference and prediction. It is inferential because we want to understand which variables are significantly associated with wildfire size, and predictive because we also want to see how well those variables help predict the size of a wildfire.

- The response variable will be the dataset’s continuous wildfire-size measure, such as burned area or another continuous fire-size variable in the dataset. We could also use the first_uc_date which is the date in which this wildfire was first under control, to see how long it usually takes to get a wildfire undercontrol given several parameters.

- I expect the most important covariates to be temperature, relative_humidity, wind_speed, fuel_type, weather_conditions_over_fire, fire_spread_rate, fire_type, and location variables such as latitude and longitude. I would also expect general_cause and year to matter, since fire behavior and wildfire severity can vary by ignition source and across years. All of this is just a hunch at this stage, and I will be using the EDA below to reach a conclusion.

- Other variables may be included to help control for differences in geography, timing, and wildfire management. For example, variables related to detection and suppression, such as detection_agent_type, initial_action_by, ia_access, and bucketing_on_fire, may help account for differences in how quickly fires were discovered and responded to, while date-related variables can help control for seasonal or temporal effects.

> ## Minor change to make the output more readable for missing values and only output those with the greatest percentage of missing values.

### Lets begin by cleanning our data before we can get into to the EDA

In [ ]:
wildfire_clean <- wildfire

# remove exact duplicate rows
wildfire_clean <- unique(wildfire_clean)

# trim whitespace and turn blank strings into NA
char_cols <- names(wildfire_clean)[sapply(wildfire_clean, is.character)]
for (col in char_cols) {
  wildfire_clean[[col]] <- trimws(wildfire_clean[[col]])
  wildfire_clean[[col]][wildfire_clean[[col]] == ""] <- NA
}

# convert known numeric columns
numeric_cols <- c(
  "year",
  "current_size",
  "latitude",
  "longitude",
  "assessment_hectares",
  "fire_spread_rate",
  "temperature",
  "relative_humidity",
  "wind_speed",
  "fire_fighting_start_size",
  "first_bh_size",
  "first_uc_size",
  "first_ex_size_perimeter"
)

numeric_cols <- intersect(numeric_cols, names(wildfire_clean))

for (col in numeric_cols) {
  wildfire_clean[[col]] <- as.numeric(wildfire_clean[[col]])
}

# convert known date/time columns
date_cols <- c(
  "fire_start_date",
  "ia_arrival_at_fire_date",
  "fire_fighting_start_date",
  "first_bh_date",
  "first_uc_date"
)

date_cols <- intersect(date_cols, names(wildfire_clean))

for (col in date_cols) {
  wildfire_clean[[col]] <- as.POSIXct(
    wildfire_clean[[col]],
    tryFormats = c(
      "%Y-%m-%d %H:%M:%S",
      "%Y-%m-%d %H:%M",
      "%Y-%m-%d",
      "%m/%d/%Y %H:%M:%S",
      "%m/%d/%Y %H:%M",
      "%m/%d/%Y"
    ),
    tz = "UTC"
  )
}

# convert categorical variables to factors
factor_cols <- c(
  "size_class",
  "fire_origin",
  "general_cause",
  "responsible_group",
  "activity_class",
  "true_cause",
  "detection_agent_type",
  "detection_agent",
  "fire_type",
  "fire_position_on_slope",
  "weather_conditions_over_fire",
  "wind_direction",
  "fuel_type",
  "initial_action_by",
  "ia_access",
  "bucketing_on_fire"
)

factor_cols <- intersect(factor_cols, names(wildfire_clean))

for (col in factor_cols) {
  wildfire_clean[[col]] <- as.factor(wildfire_clean[[col]])
}


## In the cells below I will be making a few checks and clearing those rows that have illegal values. Ideally since this data is
## from albertan government, this is already done

# set impossible values to NA
nonnegative_cols <- c(
  "current_size",
  "assessment_hectares",
  "fire_spread_rate",
  "relative_humidity",
  "wind_speed",
  "fire_fighting_start_size",
  "first_bh_size",
  "first_uc_size",
  "first_ex_size_perimeter"
)

nonnegative_cols <- intersect(nonnegative_cols, names(wildfire_clean))

for (col in nonnegative_cols) {
  wildfire_clean[[col]][wildfire_clean[[col]] < 0] <- NA
}

# relative humidity should not exceed 100
if ("relative_humidity" %in% names(wildfire_clean)) {
  wildfire_clean$relative_humidity[wildfire_clean$relative_humidity > 100] <- NA
}

# latitude and longitude range checks
if ("latitude" %in% names(wildfire_clean)) {
  wildfire_clean$latitude[wildfire_clean$latitude < -90 | wildfire_clean$latitude > 90] <- NA
}
if ("longitude" %in% names(wildfire_clean)) {
  wildfire_clean$longitude[wildfire_clean$longitude < -180 | wildfire_clean$longitude > 180] <- NA
}

# These have produced long and not great to read outputs, so i want to change these with the following overview
# summary(wildfire_clean)
# dim(wildfire_clean)

wildfire_overview = data.frame(
  variable = names(wildfire_clean),
  type = sapply(wildfire_clean, function(x) class(x)[1]),
  missing_count = sapply(wildfire_clean, function(x) sum(is.na(x))),
  missing_percentage = round(sapply(wildfire_clean, function(x) mean(is.na(x))) * 100, 1),
  unique_values = sapply(wildfire_clean, function(x) dplyr::n_distinct(x))
)

wildfire_overview |>
        select(type, missing_count, missing_percentage, unique_values) |>
        arrange(desc(missing_percentage)) |>
        head()

### Now we want to see how many missing values are there and the proportion of these

I calculated the number and proportion of missing values for each variable in the dataset. The variables with more than 20% of the data missing may need to be excluded, imputed, or interpreted with caution in the analysis.

> ## This section has a piece of code corrected so that missing_proportion is accurately called

In [ ]:
missing_table <- data.frame(
  missing_count = sapply(wildfire, function(x) sum(is.na(x))),
  missing_proportion = sapply(wildfire, function(x) mean(is.na(x)))
)

missing_table <- missing_table[order(-missing_table$missing_count), ]
missing_table[missing_table$missing_proportion > 0.20, ]

> ## Reduced junk output in this section

### Checking for class imbalance
To assess class imbalance, I examined the distribution of each categorical variable by computing category proportions. Evidence of class imbalance would be present in variables where one category accounts for a large majority of observations, and these can be identified by checking whether the largest class proportion exceeds a chosen threshold such as 70%.

In [ ]:
cat_cols <- names(wildfire)[sapply(wildfire, function(x) is.factor(x) || is.character(x))]

# for each categorical variable, compute counts and proportions
class_balance <- lapply(cat_cols, function(col) {
  tab <- table(wildfire[[col]], useNA = "ifany")
  prop <- prop.table(tab)
  
  data.frame(
    variable = col,
    category = names(tab),
    count = as.vector(tab),
    proportion = as.vector(prop)
  )
})

names(class_balance) <- cat_cols
# class_balance

# class_balance_summary = bind_rows(class_balance) %>%
#   group_by(variable) %>%
#   slice_max(order_by = proportion, n = 1, with_ties = FALSE) %>%
#   ungroup() %>%
#   mutate(proportion = round(proportion, 3)) %>%
#   arrange(desc(proportion))

# class_balance_summary       


# change slice head n, to show the top n category in each variable 
#contributing to class imbalance. Kept at n=2 for a compact output at this time.
                                   
class_balance_compact = bind_rows(class_balance) %>%
  group_by(variable) %>%
  arrange(desc(proportion), .by_group = TRUE) %>%
  slice_head(n = 2) %>%
  ungroup() %>%
  mutate(proportion = round(proportion, 3))

class_balance_compact

In [ ]:
##where actual imbalance is present in here

imbalance_summary <- data.frame(
  variable = cat_cols,
  largest_class_prop = sapply(cat_cols, function(col) {
    tab <- table(wildfire[[col]], useNA = "no")
    max(prop.table(tab))
  })
)

imbalance_summary <- imbalance_summary[order(-imbalance_summary$largest_class_prop), ]
imbalance_summary

> There is evidence of possible class imbalance in several categorical variables here. In particular, fire_position_on_slope is highly imbalanced, with one category accounting for about 72.9% of observations, and variables such as fire_type, size_class, responsible_group, and ia_access also show notable imbalance because one category makes up more than half of the dataset.

**However, fire_number is an identifier rather than a meaningful categorical variable, so it should not be interpreted in this context.**

## EDA Plot

For my EDA, I am presenting a faceted scatterplot with temperature on the x-axis, relative humidity on the y-axis, point size representing final wildfire size (current_size), point colour representing wind speed, and separate facets for general_cause. This plot is relevant because my question asks how weather and wildfire characteristics are associated with final wildfire size, and this visualization allows us to examine several important variables at the same time. It is especially useful for seeing whether larger fires tend to occur in hotter, drier, and windier conditions, and whether these patterns differ depending on the cause of the fire.

To improve readability and reduce computation time, I created the plot using a random sample of up to 500 observations. I then plotted temperature and relative humidity, used point colour to represent wind speed, used point size to represent final wildfire size, and faceted by general cause to examine how weather conditions and cause are related to wildfire size.

In [ ]:
set.seed(301)
plot_data <- wildfire %>%
  filter(
    !is.na(temperature),
    !is.na(relative_humidity),
    !is.na(wind_speed),
    !is.na(current_size),
    !is.na(general_cause)
  )

options(repr.plot.width = 12, repr.plot.height = 12)
sample_size <- min(500, nrow(plot_data))

plot_data <- plot_data %>%
  slice_sample(n = sample_size) %>%
  mutate(log_current_size = log1p(current_size)) ## i mutated size to log as it looked odd and exponential without doing this

ggplot(plot_data, aes(x = temperature, y = relative_humidity)) +
  geom_point(
    aes(size = log_current_size, colour = wind_speed),
    alpha = 0.4
  ) + facet_wrap(~ general_cause, ncol = 2) +
  labs(
    title = "Wildfire Size by Weather Conditions and Cause",
    subtitle = "Random sample of 500 wildfires",
    x = "Temperature (°C)",
    y = "Relative Humidity (%)",
    colour = "Wind Speed (km/h)",
    size = "Final estimated area burned by the wildfire (Ha)"
  ) + theme_minimal(base_size = 13) + theme(
    plot.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold", size = 12)
  )

## Concluding points observed in the EDA-

Larger wildfires seem to occur more often under lower relative (under 70%) humidity and moderate to warmer temperatures, especially in the Lightning, Incendiary, and Resident groups. The Lightning panel has by far the most observations, suggesting that this cause is very common in the data, but there is no single simple pattern showing that higher temperature or higher wind speed alone always leads to the largest fires, atleast that is not easily percievable here.

But we do see in most cases, the points are most dense and tend towards a negative correlation, so higher temperature, and lower humidity has the most causes of wildfires.

# Assignment 2- Methods and Plan & Computational Code and Output

## (1) Methods and Plan

(a) **Reviewed Question:** How are environmental, seasonal, and geographic variables associated with wildfire size in the Alberta wildfire dataset?

In Assignment 1, my question focused on how weather and wildfire-related factors might explain variation in wildfire size. For Assignment 2, I am refining that question so that it is more clearly aligned with the data and with an observational study design. In particular, I am focusing on **association rather than causation**, since this dataset is observational, it does not support strong causal conclusions. I will use `current_size` as the response variable, since it is a direct continuous measure of wildfire size and is more suitable for a regression-based analysis than a date variable such as `first_uc_date`.

(b) I propose to use a **multiple linear regression model** with a **log-transformed wildfire size response**, specifically `log(current_size)`, to address my question of interest. This method is appropriate because the response variable, wildfire size, is continuous, and the EDA suggests that it is highly right-skewed. Applying a log transformation makes the scale of the response more stable and helps reduce the influence of extremely large fires.

Multiple linear regression is also a suitable choice because my question concerns the joint relationship between wildfire size and several predictors at once, such as temperature, relative humidity, wind speed, year, general cause, and geographic variables like latitude and longitude. This method allows me to estimate how each predictor is associated with wildfire size while holding the others fixed. I chose this method because it is a standard course method, it matches the type of response variable I am studying, and it supports an interpretable analysis of how several variables are associated with wildfire size in the dataset.

(c) Several assumptions are required for this method. 

First, the observations should be approximately independent, meaning that one wildfire record should not strongly determine another. 
Second, the relationship between the predictors and the transformed response should be approximately linear on the model scale. 
Third, the variance of the residuals should be reasonably constant across fitted values. 
Fourth, the residuals should be approximately normally distributed, especially after applying the log transformation to wildfire size. 
Finally, the predictors should not be too highly correlated with one another, since strong multicollinearity can make coefficient estimates unstable and difficult to interpret.

There are also important limitations to this approach. Because the dataset is observational, the model can only identify **associations** and not causal effects. In addition, wildfire size is often influenced by factors that may not be fully measured in the dataset, so omitted variables could affect the results. Another limitation is that even after log transformation, the response may still not perfectly satisfy all regression assumptions, especially if there are extreme fires or unusual observations. Missing values may also reduce the available sample size and potentially bias the analysis if the missingness is not random.

To be sure of the fact that we wont be violating any assumptions for a multiple linear regression we will need to be checking and verifying the following after a model is fitted:
- Linearity : After fitting the multiple linear regression model, linearity can be assessed by examining residual plots, especially the residuals versus fitted values plot and component-plus-residual plots for the continuous predictors. If the relationship is approximately linear, the residuals should appear randomly scattered around 0 without a clear curved pattern. A strong curve or systematic pattern would suggest that the linear form may not be appropriate for one or more predictors.

- Independence of observations: Independence can be checked from the structure of the dataset. In this case, it can be checked by confirming that each row represents a distinct wildfire and that there are no repeated measurements or duplicated wildfire records that would create dependence between observations. If the data contain repeated or clustered observations, then the independence assumption may be questionable.

- Homoscedasticity: Homoscedasticity can be checked after fitting the model by looking at the residuals versus fitted values plot. If the assumption holds, the spread of the residuals should remain roughly constant across the fitted values instead of forming a funnel shape or another systematic pattern. A strong change in spread would suggest heteroscedasticity.

- No perfect multicollinearity: Multicollinearity can be checked after fitting the model by examining variance inflation factors (VIF) and the correlation matrix of the numeric predictors. If the predictors are not excessively collinear, the VIF values should remain at a reasonable level and no predictor should appear to be almost a linear combination of the others. Very large VIF values would indicate that the regression coefficients may be unstable and harder to interpret. We could then simply pruge one of those coefficients and re-fit.

- Normality of errors: Normality of the errors can assessed after fitting the model by examining the residual distribution, with a normal Q-Q plot or a histogram of the residuals. If the assumption is reasonable, the residuals should follow the reference line fairly closely in the Q-Q plot and appear approximately bell-shaped in the histogram. Large variation from the line would suggest that the error distribution is not close to normal.

We have to satisfy all these conditions for our multiple linear regression model to hold.

## (2) Computational Code and Output

(a) In this section, I am experimenting with several different multiple linear regression approaches for our cleaned dataset by keeping only observations with non-missing values for the response variable, `current_size`, and for the selected predictors: temperature, relative humidity, wind speed, latitude, longitude, year, and general cause. 

I will then create a log-transformed version of wildfire size, `log1p(current_size)`, because wildfire size is strongly right-skewed and the transformation helps make the regression assumptions more reasonable. After that, I will fit a multiple linear regression model using the transformed wildfire size as the response and the selected environmental, geographic, and wildfire-cause variables as predictors. Finally, I will present the model output in a single results table containing the estimated coefficients, confidence intervals, and p-values for interpretation.

In [ ]:
model_data = wildfire_clean %>%
  filter(
    !is.na(current_size),
    !is.na(temperature),
    !is.na(relative_humidity),
    !is.na(wind_speed),
    !is.na(latitude),
    !is.na(longitude),
    !is.na(year),
    !is.na(general_cause)
  ) %>%
  mutate(
    log_current_size = log1p(current_size),
    general_cause = factor(general_cause)
  )

wildfire_size_model = lm(
  log_current_size ~ temperature + relative_humidity + wind_speed +
    latitude + longitude + year + general_cause,
  data = model_data
)
#wildfire_size_model

In [ ]:
vif_values = car::vif(selected_model)

# car::vif() returns a matrix when factors are present
if (is.matrix(vif_values)) {
  vif_table = data.frame(
    term = rownames(vif_values),
    GVIF = round(vif_values[, "GVIF"], 3),
    Df = vif_values[, "Df"],
    GVIF_adj = round(vif_values[, "GVIF^(1/(2*Df))"], 3)
  )
} else {
  vif_table = data.frame(
    term = names(vif_values),
    VIF = round(vif_values, 3)
  )
}

vif_table

(b) Table of Results

The table above reports the estimated coefficients, 95% confidence intervals, and p-values from the multiple linear regression model with log(current_size) as the response variable. I have also arranged the absolute values of the estimate in descending order so we know what are the most relevant coefficients we can make immediate interpretations about. It should be obvious why the absolute value of the estimate is considered- we dont want the coefficients with the negative sign to end up at the bottom, just those close to 0 as those have very little impact on the value of the response.

In [ ]:
results_table = tidy(wildfire_size_model, conf.int = TRUE) %>%
  filter(term != "(Intercept)") %>%
  mutate(
    estimate = round(estimate, 3),
    conf.low = round(conf.low, 3),
    conf.high = round(conf.high, 4),
    p.value = signif(p.value, 3)
  ) %>%
  # filter(p.value <= 0.05) %>%
  select(term, estimate, conf.low, conf.high, p.value) %>%
  arrange(desc(abs(estimate)))

results_table

(c) After adjusting for the other variables in the model, wildfires with general cause **Under Investigation**, and **Lightning** showed the strongest positive associations with log-transformed wildfire size relative to the reference cause category, so the cause has the greatest impact on the total area burned. 

Among the continuous predictors, **higher latitude**, **higher wind speed**, and **later years** were associated with larger wildfire size, while **higher relative humidity** was associated with smaller wildfire size; temperature and longitude were also positive but much weaker in magnitude. 

Several cause categories, such as **Prescribed Fire**, **Railroad**, and **Undetermined**, did not show clear evidence of association because their confidence intervals included 0. 

Overall, these results suggest that both wildfire cause and environmental conditions are associated with wildfire size in this dataset, although the findings should be interpreted as associations rather than causal effects.

### AI Use Disclosure
> AI was used by me to debug the code to find the proportion of class imbalances as i struggled to get that code working properly myself. I also used it to debug my plot code which had issues with faceting, and sizing. I had incorrectly used faceting earlier and the AI gave me alternative code to use which I had implemented myself here.
> 
> For assignment 2 sections, AI was not consulted to create any of the new pieces of code.

All other work is my own, all the writing done, the idea and concept is my own.

### Additional Disclosure
> Some of the EDA, cleaning and wrangling done here is inspired by methods taught to me in another course- DSCI 320, I merely repurposed some of that code to use in this context
>
> For assignment 2, only materials from STAT 301 was considered in terms of the assumptions for a multiple linear regression model.